# Ablation of the provision-network features, by category (A19)

Removing all provision-network features costs only about 0.008 test F1 overall (`08`). The taxonomy argument says
these features should matter most where a wallet's funding links it to other interactors (IxI, IxE). This notebook
tests that per category.

Configurations (LightGBM, parameters from `02`, model seed 42, 10 group splits in `sp.SPLIT_SEEDS`, all on one
machine): **All features**; **Without all provision-network families** (gas provider, gas provision tree, provision
chain); and each of those three families removed alone. Per category (IxL, IxI, IxE) and overall: the paired
difference All − Without in F1, recall and AUROC, with mean ± SD, wins out of 10, and the corrected resampled t-test
(as in `17`). Each model's threshold is chosen on validation.

**Results**: `results/19_ablation_by_category.json`.

## Setup

In [1]:
import os, platform, time, warnings
import numpy as np
import pandas as pd
from scipy import stats
from sklearn.metrics import average_precision_score, f1_score, roc_auc_score
import sybil_pipeline as sp
warnings.filterwarnings('ignore')
pd.set_option('display.width', 220)
assert not sp.CODE_COMMIT.endswith('-dirty'), f'Run from a clean working tree (CODE_COMMIT={sp.CODE_COMMIT})'
print('Code commit:', sp.CODE_COMMIT)
PLATFORM = dict(machine=platform.machine(), processor=platform.processor(), system=platform.system(),
                python=platform.python_version(), cpu_count=os.cpu_count(), n_jobs=sp.N_JOBS)
print('Platform:', PLATFORM)
DATA_DIR = './data'
_, LGBM_P = sp.load_selected_params()
FAM = {k: [f for f in v if f in sp.FEATS] for k, v in sp.FEATURE_FAMILIES.items()}
NETWORK = ['Gas provider', 'Gas provision tree', 'Provision chain']
CONFIGS = {'All features': sp.FEATS,
           'Without all provision-network families': [f for f in sp.FEATS if not any(f in FAM[k] for k in NETWORK)]}
CONFIGS.update({f'Without {k}': [f for f in sp.FEATS if f not in FAM[k]] for k in NETWORK})
for k, v in CONFIGS.items():
    print(f'{k}: {len(v)} features')
df, _, _ = sp.build_master_df(DATA_DIR, verbose=False)
CATS = ['IxL', 'IxI', 'IxE']

Code commit: cdb7aed
Platform: {'machine': 'x86_64', 'processor': 'x86_64', 'system': 'Linux', 'python': '3.11.15', 'cpu_count': 4, 'n_jobs': 4}
All features: 62 features
Without all provision-network families: 31 features
Without Gas provider: 51 features
Without Gas provision tree: 45 features
Without Provision chain: 59 features


## Fits

In [2]:
rows, ratios = [], []
for s in sp.SPLIT_SEEDS:
    t0 = time.time()
    S = sp.make_splits(df, sp.FEATS, method='group', seed=s)
    ratios.append(len(S['idx_test']) / len(S['idx_train']))
    cat = df.loc[S['idx_test'], 'category']
    for name, feats in CONFIGS.items():
        r = sp.fit_lgbm(S, LGBM_P, seeds=(42,), feats=feats, verbose=False)
        thr = sp.best_f1_threshold(S['y_val'], r['val'])
        yhat = (r['test'] >= thr).astype(int)
        y = S['y_test'].to_numpy()
        rows.append(dict(split_seed=s, config=name, category='All', n=len(y), sybil=int(y.sum()), threshold=thr,
                         f1=f1_score(y, yhat), recall=float(yhat[y == 1].mean()), auroc=roc_auc_score(y, r['test']),
                         ap=average_precision_score(y, r['test'])))
        bc = sp.metrics_by_category(S['y_test'], r['test'], cat, thr)
        assert bc['n'].sum() == len(y) and bc['sybil'].sum() == y.sum()          # categories sum to the test totals
        rows += [dict(split_seed=s, config=name, threshold=thr, **{k: x[k] for k in ['category', 'n', 'sybil', 'f1', 'recall', 'auroc']})
                 for x in bc.to_dict('records')]
    print(f'split {s} done ({time.time() - t0:.0f}s)', flush=True)
res = pd.DataFrame(rows)
assert res.groupby('config')['split_seed'].nunique().eq(10).all()

split 42 done (272s)


split 1 done (289s)


split 2 done (287s)


split 3 done (264s)


split 4 done (245s)


split 5 done (262s)


split 6 done (242s)


split 7 done (270s)


split 8 done (261s)


split 9 done (261s)


### Check against `08` (overall "All features")

In [3]:
r08 = pd.DataFrame(sp.load_results('08_ablation_families')['per_split'])
r08 = r08[r08['config'] == 'All features'].set_index('split_seed')['test_f1']
mine = res[(res['config'] == 'All features') & (res['category'] == 'All')].set_index('split_seed')['f1']
MATCHES_08 = bool(((mine - r08).abs() < 1e-12).all())
print(f"All features, overall test F1 vs 08: max |difference| {(mine - r08).abs().max():.4f}; identical: {MATCHES_08}"
      + ('' if MATCHES_08 else f"  (08 ran on another machine; this run: {PLATFORM['machine']} / {PLATFORM['system']})"))

All features, overall test F1 vs 08: max |difference| 0.0000; identical: True


## Paired differences (All features − Without), per category

In [4]:
RATIO = float(np.mean(ratios))   # unique test rows / unique training rows, mean over splits
N = 10
def corrected_t(d):
    v = np.var(d, ddof=1)
    if v == 0:
        return np.nan, np.nan
    t = np.mean(d) / np.sqrt((1 / N + RATIO) * v)
    return float(t), float(2 * stats.t.sf(abs(t), N - 1))

base = res[res['config'] == 'All features'].set_index(['split_seed', 'category'])
out = []
for name in CONFIGS:
    if name == 'All features':
        continue
    other = res[res['config'] == name].set_index(['split_seed', 'category'])
    for c in ['All'] + CATS:
        for m in ['f1', 'recall', 'auroc']:
            d = (base[m] - other[m]).xs(c, level='category').loc[sp.SPLIT_SEEDS].to_numpy()
            t, p = corrected_t(d)
            out.append(dict(without=name, category=c, metric=m, all_mean=float(base[m].xs(c, level='category').mean()),
                            without_mean=float(other[m].xs(c, level='category').mean()),
                            diff_mean=float(d.mean()), diff_sd=float(d.std(ddof=1)), all_higher_n=int((d > 0).sum()),
                            t_corrected=t, p_corrected=p))
diffs = pd.DataFrame(out)
print(f'n_test / n_train (unique rows) = {RATIO:.4f}')
for name in diffs['without'].unique():
    print(f'\n=== All features − {name}')
    t = diffs[diffs['without'] == name].pivot(index='category', columns='metric', values=['diff_mean', 'diff_sd', 'all_higher_n', 'p_corrected'])
    print(t.loc[['All'] + CATS].round(4).to_string())

n_test / n_train (unique rows) = 0.6122

=== All features − Without all provision-network families
         diff_mean                 diff_sd                 all_higher_n             p_corrected                
metric       auroc      f1  recall   auroc      f1  recall        auroc   f1 recall       auroc      f1  recall
category                                                                                                       
All         0.0002  0.0079  0.0220  0.0012  0.0076  0.0243          6.0  7.0    8.0      0.8857  0.2484  0.3115
IxL         0.0004  0.0027  0.0165  0.0005  0.0047  0.0230          8.0  8.0    7.0      0.3036  0.5072  0.4169
IxI         0.0008  0.1137  0.1049  0.0133  0.1124  0.0976          6.0  9.0    9.0      0.9473  0.2614  0.2347
IxE         0.0114 -0.0379 -0.0297  0.0075  0.0444  0.0314          9.0  3.0    1.0      0.1037  0.3382  0.2902

=== All features − Without Gas provider
         diff_mean                 diff_sd                 all_higher_n     

### Mean per configuration and category

In [5]:
means = res.groupby(['config', 'category'])[['f1', 'recall', 'auroc']].agg(['mean', 'std'])
print(means.round(4).to_string())

                                                        f1          recall           auroc        
                                                      mean     std    mean     std    mean     std
config                                 category                                                   
All features                           All          0.7131  0.0060  0.7104  0.0223  0.9682  0.0018
                                       IxE          0.2420  0.0654  0.1479  0.0446  0.9099  0.0157
                                       IxI          0.4915  0.0872  0.3482  0.0779  0.9254  0.0103
                                       IxL          0.7395  0.0041  0.7712  0.0227  0.9752  0.0012
                                       No provider  0.7158  0.1279  0.6261  0.1483  0.9679  0.0279
Without Gas provider                   All          0.7081  0.0068  0.7000  0.0218  0.9674  0.0014
                                       IxE          0.2373  0.0678  0.1453  0.0463  0.9009  0.0156
          

## Save results

In [6]:
sp.save_results([], '19_ablation_by_category', extra=dict(
    configs={k: len(v) for k, v in CONFIGS.items()}, families=FAM, params=LGBM_P, model_seed=42,
    per_split=res.to_dict('records'), diffs=diffs.to_dict('records'),
    means={f'{c}|{k}': {f'{m}_{a}': float(means.loc[(c, k), (m, a)]) for m in ['f1', 'recall', 'auroc'] for a in ['mean', 'std']}
           for c, k in means.index},
    test_method=dict(headline='corrected resampled t-test (Nadeau and Bengio 2003), df = 9', n_test_over_n_train=RATIO),
    all_features_matches_08=MATCHES_08, platform=PLATFORM))

Saved results/19_ablation_by_category.json
